# AIS 데이터 분석
부산 실해역 테스트 데이터

In [13]:
import pandas as pd

FILE_PATH = 'AIS_부산 실해역 테스트 데이터.xlsx'

## 1. 파일 구조 확인

In [14]:
# 시트 목록 확인
xl = pd.ExcelFile(FILE_PATH)
print('시트 목록:', xl.sheet_names)

시트 목록: ['26.06.09', '26.06.10(~07 59)', '26.06.10(08 00 ~ 15 06)']


In [15]:
# 전체 시트 읽기 (헤더 없음)
sheets = pd.read_excel(FILE_PATH, sheet_name=None, header=None)

for name, df in sheets.items():
    print(f'\n=== 시트: {name} ===')
    print(f'행: {df.shape[0]}, 열: {df.shape[1]}')
    print('샘플:', df.iloc[0].tolist())


=== 시트: 26.06.09 ===
행: 684665, 열: 2
샘플: ['20260609 17:56:42.9433', '!AIVDM,1,1,3,B,36Sf4MPP@Ta>qd`D5PSri9eD21q0,0*52']

=== 시트: 26.06.10(~07 59) ===
행: 905769, 열: 2
샘플: ['20260610 00:00:00.2866', '!AIVDM,1,1,6,A,16S`e90P?w<tSF0l4Q@>41gp1PRL,0*27']

=== 시트: 26.06.10(08 00 ~ 15 06) ===
행: 435165, 열: 2
샘플: ['20260610 08:00:00.0091', '!AIVDM,1,1,9,B,16TktR50009>tMTD5dieHET00@0f,0*3B']


## 2. 첫 번째 시트 상세 확인

In [16]:
# 첫 번째 시트를 메인 데이터로 사용 (헤더 없음)
first_sheet = xl.sheet_names[0]
df = pd.read_excel(FILE_PATH, sheet_name=first_sheet, header=None)
df.columns = ['timestamp', 'raw_msg']

print(f'시트: {first_sheet}')
print(f'크기: {df.shape}')
df.head(10)

시트: 26.06.09
크기: (684665, 2)


,timestamp,raw_msg
0,20260609 17:56:42.9433,"!AIVDM,1,1,3,B,36Sf4MPP@Ta>qd`D5PSri9eD21q0,0*52"
1,20260609 17:56:42.9433,"$AIVSI,0,3,085643.550062,1633,-73,41*4A"
2,20260609 17:56:42.9700,"!AIVDM,1,1,4,A,16TC3M80h09>gmFD68AV11mD08Je,0*34"
3,20260609 17:56:42.9700,"$AIVSI,0,4,085643.576728,1634,-62,52*45"
4,20260609 17:56:43.0767,"!AIVDM,1,1,5,A,16Sk9MPP00a>`idD4qk6<wwD28Ji,0*67"
5,20260609 17:56:43.0767,"$AIVSI,0,5,085643.683395,1638,-74,40*47"
6,20260609 17:56:43.2101,"!AIVDM,1,1,6,A,36SfPqPOhsa>td<D5:R8:7?B01w@,0*11"
7,20260609 17:56:43.2101,"$AIVSI,0,6,085643.816791,1643,-62,52*4E"
8,20260609 17:56:43.2633,"!AIVDM,1,1,7,B,16Sh9ASw@09?41:D6W:4@ngF0<4h,0*4F"
9,20260609 17:56:43.2633,"$AIVSI,0,7,085643.870021,1645,-86,29*43"


In [17]:
# 컬럼별 데이터 타입 및 null 현황
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 684665 entries, 0 to 684664
Data columns (total 2 columns):
 #   Column     Non-Null Count   Dtype
---  ------     --------------   -----
 0   timestamp  684665 non-null  str  
 1   raw_msg    684665 non-null  str  
dtypes: str(2)
memory usage: 10.4 MB


In [18]:
# 샘플 데이터 확인 (각 컬럼 첫 번째 유효값)
df.head(5)

,timestamp,raw_msg
0,20260609 17:56:42.9433,"!AIVDM,1,1,3,B,36Sf4MPP@Ta>qd`D5PSri9eD21q0,0*52"
1,20260609 17:56:42.9433,"$AIVSI,0,3,085643.550062,1633,-73,41*4A"
2,20260609 17:56:42.9700,"!AIVDM,1,1,4,A,16TC3M80h09>gmFD68AV11mD08Je,0*34"
3,20260609 17:56:42.9700,"$AIVSI,0,4,085643.576728,1634,-62,52*45"
4,20260609 17:56:43.0767,"!AIVDM,1,1,5,A,16Sk9MPP00a>`idD4qk6<wwD28Ji,0*67"


## 3. VDM / VSI 페어링 및 멀티파트 조립

## 4. Type 1/3 메시지 파싱

In [19]:
# 데이터 품질 사전 검증
issues = []

vdm_buffer_check = []

for idx, row in df.iterrows():
    msg = str(row['raw_msg']).strip()
    ts = row['timestamp']

    if msg.startswith('!'):
        parts = msg.split(',')
        total = int(parts[1])
        part_num = int(parts[2])
        seq_id = parts[3]

        if total == 1:
            # 단일 파트인데 버퍼에 잔여물이 있으면 이전 멀티파트가 미완성
            if vdm_buffer_check:
                issues.append({
                    'row': idx,
                    'type': '멀티파트 미완성',
                    'detail': f'버퍼에 {len(vdm_buffer_check)}개 조각 남은 상태에서 새 단일파트 시작',
                    'msg': vdm_buffer_check[0][1]
                })
                vdm_buffer_check = []
            vdm_buffer_check.append((ts, msg, part_num, total))
            vdm_buffer_check = []  # 단일파트는 바로 완료
        else:
            if part_num == 1:
                # 새 멀티파트 시작 — 버퍼에 잔여물 있으면 이전 것 미완성
                if vdm_buffer_check:
                    issues.append({
                        'row': idx,
                        'type': '멀티파트 미완성',
                        'detail': f'버퍼에 {len(vdm_buffer_check)}개 조각 남은 상태에서 새 멀티파트 시작',
                        'msg': vdm_buffer_check[0][1]
                    })
                vdm_buffer_check = [(ts, msg, part_num, total)]
            else:
                if not vdm_buffer_check:
                    issues.append({
                        'row': idx,
                        'type': '멀티파트 앞조각 없음',
                        'detail': f'part {part_num}/{total} 인데 앞조각 없음',
                        'msg': msg
                    })
                elif part_num != vdm_buffer_check[-1][2] + 1:
                    issues.append({
                        'row': idx,
                        'type': '멀티파트 순서 오류',
                        'detail': f'예상 part {vdm_buffer_check[-1][2]+1}, 실제 part {part_num}',
                        'msg': msg
                    })
                else:
                    vdm_buffer_check.append((ts, msg, part_num, total))

                if vdm_buffer_check and part_num == total:
                    vdm_buffer_check = []

    elif msg.startswith('$AIVSI'):
        pass  # VSI는 별도 검증 불필요

# 버퍼에 잔여물 남아있으면 마지막 멀티파트 미완성
if vdm_buffer_check:
    issues.append({
        'row': 'EOF',
        'type': '멀티파트 미완성 (파일 끝)',
        'detail': f'{len(vdm_buffer_check)}개 조각만 있고 끝남',
        'msg': vdm_buffer_check[0][1]
    })

if issues:
    print(f'문제 발견: {len(issues)}건')
    pd.DataFrame(issues)
else:
    print('이상 없음 — 모든 멀티파트 정상 조립 가능')

이상 없음 — 모든 멀티파트 정상 조립 가능


In [20]:
def decode_msg_type(payload):
    """AIS payload 첫 문자에서 메시지 타입 추출"""
    try:
        val = ord(payload[0]) - 48
        if val > 39:
            val -= 8
        return val
    except:
        return None

def parse_vdm_fields(msg):
    """VDM 문장에서 필드 파싱"""
    parts = msg.split(',')
    return {
        'total': int(parts[1]),
        'part_num': int(parts[2]),
        'seq_id': parts[3],
        'payload': parts[5],
    }

records = []
vdm_buffer = []  # 멀티파트 조각 임시 저장

for _, row in df.iterrows():
    msg = str(row['raw_msg']).strip()
    ts = row['timestamp']

    if msg.startswith('!'):
        p = parse_vdm_fields(msg)
        vdm_buffer.append((ts, msg, p))

        if p['part_num'] == p['total']:
            # 모든 조각 모임 → 한 행으로 합치기
            combined_raw = '|'.join(m for _, m, _ in vdm_buffer)
            first_ts = vdm_buffer[0][0]
            full_payload = ''.join(fp['payload'] for _, _, fp in vdm_buffer)
            records.append({
                'timestamp': first_ts,
                'msg_num': decode_msg_type(full_payload),
                'raw_msg': combined_raw,
                'vsi': None
            })
            vdm_buffer = []

    elif msg.startswith('$AIVSI'):
        # 바로 직전 완성된 VDM 행에 VSI 연결
        if records:
            records[-1]['vsi'] = msg

result_df = pd.DataFrame(records, columns=['timestamp', 'msg_num', 'raw_msg', 'vsi'])
print(f'총 메시지 수: {len(result_df)}')
print(f'msg_num 분포:\n{result_df["msg_num"].value_counts().sort_index()}')
result_df.head(10)

총 메시지 수: 336193
msg_num 분포:
msg_num
1     289753
3      25209
4       2375
5      11148
6        818
7        214
8       1202
12         2
15       419
18       585
19       432
20      2519
21      1138
24       379
Name: count, dtype: int64


,timestamp,msg_num,raw_msg,vsi
0,20260609 17:56:42.9433,3,"!AIVDM,1,1,3,B,36Sf4MPP@Ta>qd`D5PSri9eD21q0,0*52","$AIVSI,0,3,085643.550062,1633,-73,41*4A"
1,20260609 17:56:42.9700,1,"!AIVDM,1,1,4,A,16TC3M80h09>gmFD68AV11mD08Je,0*34","$AIVSI,0,4,085643.576728,1634,-62,52*45"
2,20260609 17:56:43.0767,1,"!AIVDM,1,1,5,A,16Sk9MPP00a>`idD4qk6<wwD28Ji,0*67","$AIVSI,0,5,085643.683395,1638,-74,40*47"
3,20260609 17:56:43.2101,3,"!AIVDM,1,1,6,A,36SfPqPOhsa>td<D5:R8:7?B01w@,0*11","$AIVSI,0,6,085643.816791,1643,-62,52*4E"
4,20260609 17:56:43.2633,1,"!AIVDM,1,1,7,B,16Sh9ASw@09?41:D6W:4@ngF0<4h,0*4F","$AIVSI,0,7,085643.870021,1645,-86,29*43"
5,20260609 17:56:43.4232,1,"!AIVDM,1,1,8,B,16S`HV0000a>hn`D6=DTliK>0D4s,0*12","$AIVSI,0,8,085644.030104,1651,-77,37*45"
6,20260609 17:56:43.4496,1,"!AIVDM,1,1,9,B,16UF0>?0009>brTD5:h5cpEH0@Il,0*17","$AIVSI,0,9,085644.056749,1652,-74,40*4B"
7,20260609 17:56:43.4770,1,"!AIVDM,1,1,0,A,11mg=5@P009>jQhD5D@DbOwIp0Rm,0*0D","$AIVSI,0,0,085644.084416,1653,-77,37*46"
8,20260609 17:56:43.5565,1,"!AIVDM,1,1,1,A,16SbFOQ000a>maBD5DBP02S@28Ip,0*5D","$AIVSI,0,1,085644.163437,1656,-71,43*4C"
9,20260609 17:56:43.6106,1,"!AIVDM,1,1,2,B,17q<5rHP019>l:pD5Dhm3wwH08Ir,0*5B","$AIVSI,0,2,085644.216728,1658,-68,46*40"


In [ ]:
from pyais import decode

TYPE_1_3_FIELDS = [
    'msg_type', 'repeat', 'mmsi', 'status', 'turn', 'speed',
    'accuracy', 'lon', 'lat', 'course', 'heading', 'second',
    'maneuver', 'raim', 'radio'
]

def parse_vdm_channel(raw_msg):
    try:
        return raw_msg.split('|')[0].split(',')[4]
    except:
        return None

def parse_vsi_fields(vsi):
    try:
        parts = vsi.split(',')
        toa = parts[3]  # 085643.550062
        vsi_hour   = int(toa[0:2])
        vsi_minute = int(toa[2:4])
        vsi_second = float(toa[4:])
        return {
            'vsi_ui':       int(parts[1]),
            'vsi_link':     int(parts[2]),
            'vsi_hour':     vsi_hour,
            'vsi_minute':   vsi_minute,
            'vsi_second':   vsi_second,
            'vsi_slot':     int(parts[4]),
            'vsi_strength': int(parts[5]),
            'vsi_snr':      int(parts[6].split('*')[0]),
        }
    except:
        return {k: None for k in ['vsi_ui','vsi_link','vsi_hour','vsi_minute','vsi_second','vsi_slot','vsi_strength','vsi_snr']}

def parse_comm_state(radio, msg_type):
    r = int(radio)
    sync_state = (r >> 17) & 0x3
    if msg_type == 1:
        return {
            'sync_state':     sync_state,
            'slot_timeout':   (r >> 14) & 0x7,
            'sub_message':    r & 0x3FFF,
            'slot_increment': None,
            'num_slots':      None,
            'keep_flag':      None,
        }
    elif msg_type == 3:
        return {
            'sync_state':     sync_state,
            'slot_timeout':   None,
            'sub_message':    None,
            'slot_increment': (r >> 5) & 0x1FFF,
            'num_slots':      (r >> 2) & 0x7,
            'keep_flag':      r & 0x1,
        }
    return {k: None for k in ['sync_state','slot_timeout','sub_message','slot_increment','num_slots','keep_flag']}

def parse_type_1_3(row):
    raw_msg = row['raw_msg']
    vsi     = row['vsi']
    try:
        parts   = [p.encode() for p in raw_msg.split('|')]
        decoded = decode(*parts).asdict()
        result  = {'timestamp': row['timestamp']}
        result['vdm_channel'] = parse_vdm_channel(raw_msg)
        result.update(parse_vsi_fields(vsi))
        result.update({f: decoded.get(f) for f in TYPE_1_3_FIELDS})
        result.update(parse_comm_state(decoded['radio'], decoded['msg_type']))
        result['raw_msg'] = raw_msg
        result['vsi']     = vsi
        return result
    except:
        all_keys = (
            ['timestamp', 'vdm_channel'] +
            ['vsi_ui','vsi_link','vsi_hour','vsi_minute','vsi_second','vsi_slot','vsi_strength','vsi_snr'] +
            TYPE_1_3_FIELDS +
            ['sync_state','slot_timeout','sub_message','slot_increment','num_slots','keep_flag'] +
            ['raw_msg','vsi']
        )
        return {k: None for k in all_keys}

mask      = result_df['msg_num'].isin([1, 3])
parsed    = result_df[mask].apply(parse_type_1_3, axis=1)
df_type13 = pd.DataFrame(parsed.tolist())

print(f'Type 1/3 메시지 수: {len(df_type13)}')
df_type13.head(10)

In [29]:
# Type 1/3 파싱 결과 엑셀 저장
OUTPUT_PATH = 'AIS_type13_parsed1.xlsx'
df_type13.to_excel(OUTPUT_PATH, index=False)
print(f'저장 완료: {OUTPUT_PATH} ({len(df_type13)}행)')

저장 완료: AIS_type13_parsed1.xlsx (314962행)
